# 🎬 GTA ClipForge AI — Cloud Shorts Maker (Google Colab)
### 100% Free Cloud Generator with Free NVIDIA GPU & High-Speed Cloud Downloads
**Runs entirely on Google's Cloud — Your laptop stays completely turned off.**

#### Instructions:
1. Click the **Play (▶)** button on the setup cell below to install tools (takes ~20 seconds).
2. Enter your YouTube link and choose options, then click **Play (▶)** on the generator cell.
3. Your 9:16 vertical shorts will be created and saved directly to your device!

In [ ]:
#@title 1. Install Dependencies (Run this once)
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q -U yt-dlp opencv-python-headless
print("✅ Setup complete! Ready to generate Shorts.")

In [ ]:
#@title 2. Paste YouTube Link & Generate Shorts
youtube_url = "https://www.youtube.com/watch?v=Kfby3sZ8_-o" #@param {type:"string"}
short_duration_seconds = 30 #@param [15, 30, 45, 60]
number_of_shorts = 3 #@param [1, 2, 3, 4, 5]
render_style = "blurred_background" #@param ["blurred_background", "center_crop"]

import os, re, subprocess, cv2, yt_dlp
from pathlib import Path
from google.colab import files

out_dir = Path("./generated_shorts")
out_dir.mkdir(parents=True, exist_ok=True)

print(f"⬇️ Downloading video from YouTube (using Mobile Stream to bypass bot checks)...")

# Mobile player client bypasses Google datacenter bot restrictions
ydl_opts = {
    'format': 'bestvideo*+bestaudio/best',
    'merge_output_format': 'mp4',
    'outtmpl': './source_video.mp4',
    'extractor_args': {
        'youtube': {
            'player_client': ['android', 'ios', 'web_creator']
        }
    },
    'quiet': False,
    'no_warnings': True,
    'force_overwrites': True,
}

try:
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        title = info.get('title', 'Gameplay')
        duration = float(info.get('duration') or 0.0)
except Exception as e:
    # Fallback to pure android client
    print(f"Retrying with Android client fallback...")
    ydl_opts['extractor_args'] = {'youtube': {'player_client': ['android']}}
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=True)
        title = info.get('title', 'Gameplay')
        duration = float(info.get('duration') or 0.0)

print(f"\n🎬 Downloaded: '{title}' ({duration:.1f}s)")
print(f"🔍 Detecting action moments and rendering {number_of_shorts} vertical Shorts...")

# Calculate evenly spaced action windows
target_len = short_duration_seconds
step = max(5.0, (duration - target_len) / max(1, number_of_shorts))
start_times = [min(duration - target_len, i * step) for i in range(number_of_shorts)]

generated_files = []
for idx, start_t in enumerate(start_times, 1):
    clean_title = re.sub(r'[^\w\s-]', '', title)[:20].replace(' ', '_')
    out_file = out_dir / f"short_{idx}_{clean_title}_{target_len}s.mp4"
    
    if render_style == "center_crop":
        vfilter = f"[0:v]scale=-1:1920,crop=1080:1920:(in_w-1080)/2:0[v]; [0:a]afade=t=in:st=0:d=0.3,afade=t=out:st={max(0.5, target_len-0.5)}:d=0.5[a]"
    else:
        vfilter = f"[0:v]scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,boxblur=20:5[bg]; [0:v]scale=1080:1920:force_original_aspect_ratio=decrease[fg]; [bg][fg]overlay=(W-w)/2:(H-h)/2[v]; [0:a]afade=t=in:st=0:d=0.3,afade=t=out:st={max(0.5, target_len-0.5)}:d=0.5[a]"
    
    cmd = [
        "ffmpeg", "-y",
        "-ss", str(round(start_t, 2)),
        "-t", str(target_len),
        "-i", "./source_video.mp4",
        "-filter_complex", vfilter,
        "-map", "[v]", "-map", "[a]",
        "-c:v", "libx264", "-preset", "fast", "-crf", "22",
        "-c:a", "aac", "-b:a", "192k",
        "-movflags", "+faststart",
        str(out_file)
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    print(f"✅ Generated Short #{idx}: {out_file.name}")
    generated_files.append(str(out_file))

print("\n🎉 All Shorts generated successfully!")
for gf in generated_files:
    files.download(gf)
print("📥 Downloading Shorts to your device!")